# Импортируем библиотеки

In [146]:
import requests
import pandas as pd
import time


In [147]:
count_text_lang = 200
time.sleep(1) 

In [148]:
headers = {
    "User-Agent": "LanguageDatasetParser/1.0 (student project; contact@example.com)"
}


In [149]:
LANGUAGES = {
    "English":    "en",
    "French":     "fr",
    "Spanish":    "es",
    "Portugeese": "pt",
    "Italian":    "it",
    "Russian":    "ru",
    "Sweedish":   "sv",
    "Malayalam":  "ml",
    "Dutch":      "nl",
    "Arabic":     "ar",
    "Turkish":    "tr",
    "German":     "de",
    "Tamil":      "ta",
    "Danish":     "da",
    "Kannada":    "kn",
    "Greek":      "el",
    "Hindi":      "hi",
}


Получает список случайных названий статей с Wikipedia

In [150]:
def get_random_article_titles(lang_code, count):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
 
    params = {
        "action":      "query",
        "list":        "random",       
        "rnnamespace": 0,              
        "rnlimit":     count,          
        "format":      "json",
    }
 
    response = requests.get(url, params=params, headers=HEADERS, timeout=10)
    data = response.json()
    print(response.status_code, response.text[:200])
    titles = [item["title"] for item in data["query"]["random"]]
    return titles


Получает текст статьи Wikipedia по её названию.

In [151]:
def get_article_text(lang_code, title):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
 
    params = {
        "action":      "query",
        "prop":        "extracts",     
        "exintro":     True,           
        "explaintext": True,           
        "titles":      title,
        "format":      "json",
    }
 
    response = requests.get(url, params=params, headers=headers, timeout=10)
    data = response.json()
 
    pages = data["query"]["pages"]
    page = next(iter(pages.values()))  
 
    text = page.get("extract", "")
    return text.strip()


  Разбивает текст на предложения по точке.
    Убирает слишком короткие предложения (меньше 30 символов).

In [152]:
def split_into_sentences(text):

    sentences = text.split(".")
 
    result = []
    for sentence in sentences:
        sentence = sentence.strip()
        if len(sentence) > 30:  
            result.append(sentence)
 
    return result


Собирает тексты для одного языка.

In [153]:
def parse_language(language_name, lang_code, texts_count):
    print(f"\n[{language_name}] Начинаем сбор текстов...")
 
    collected_texts = []
 
    try:
        titles = get_random_article_titles(lang_code, texts_count)
    except Exception as e:
        print(f"[{language_name}] Ошибка при получении списка статей: {e}")
        return []
    for i, title in enumerate(titles):
        try:
            text = get_article_text(lang_code, title)
 
            if not text:
                print(f"  [{i+1}/{len(titles)}] '{title}' — пустая статья, пропускаем")
                continue
 
            sentences = split_into_sentences(text)
            for sentence in sentences:
                collected_texts.append({
                    "Text":     sentence,
                    "Language": language_name
                })
 
            print(f"  [{i+1}/{len(titles)}] '{title}' — получено {len(sentences)} предложений")
 
            time.sleep(0.3)
 
        except Exception as e:
            print(f"  [{i+1}/{len(titles)}] '{title}' — ошибка: {e}")
            continue
 
    print(f"[{language_name}] Итого собрано: {len(collected_texts)} текстов")
    return collected_texts


In [154]:
file  = 'parsed.csv'
def main():
 
    all_texts = []  
 
    for language_name, lang_code in LANGUAGES.items():
        texts = parse_language(language_name, lang_code, count_text_lang)
        all_texts.extend(texts)
 
        df_temp = pd.DataFrame(all_texts)
        df_temp.to_csv(file, index=False, encoding="utf-8")
        print(f"  Промежуточное сохранение: {len(all_texts)} текстов в {file}")
 
    df = pd.DataFrame(all_texts)
 
    print("Сбор завершён!")
    print(f"Всего текстов: {len(df)}")
    print("\nРаспределение по языкам:")
    print(df["Language"].value_counts().to_string())
    print("=" * 50)
 
    df.to_csv(file, index=False, encoding="utf-8")
    print(f"\nФайл сохранён: {file}")


In [155]:
main()


[English] Начинаем сбор текстов...
[English] Ошибка при получении списка статей: Expecting value: line 1 column 1 (char 0)
  Промежуточное сохранение: 0 текстов в parsed.csv

[French] Начинаем сбор текстов...
[French] Ошибка при получении списка статей: Expecting value: line 1 column 1 (char 0)
  Промежуточное сохранение: 0 текстов в parsed.csv

[Spanish] Начинаем сбор текстов...
[Spanish] Ошибка при получении списка статей: HTTPSConnectionPool(host='es.wikipedia.org', port=443): Max retries exceeded with url: /w/api.php?action=query&list=random&rnnamespace=0&rnlimit=200&format=json (Caused by NewConnectionError("HTTPSConnection(host='es.wikipedia.org', port=443): Failed to establish a new connection: [Errno 101] Network is unreachable"))
  Промежуточное сохранение: 0 текстов в parsed.csv

[Portugeese] Начинаем сбор текстов...
200 {"batchcomplete":"","continue":{"rncontinue":"0.174307345954|0.174460738571|2742350|0","continue":"-||"},"query":{"random":[{"id":860654,"ns":0,"title":"Alb

In [159]:
df = pd.read_csv('parsed.csv')

In [160]:
df.head(500)

,Text,Language
0,"Alberto Mussa (Rio de Janeiro, 28 de junho de ...",Portugeese
1,Chamalycaeus takahashii é uma espécie de gastr...,Portugeese
2,Zoombombing ou invasão Zoom é uma invasão incô...,Portugeese
3,"Comumente, em invasões deste tipo, a sessão d...",Portugeese
4,Ações desse tipo fizeram com que o FBI incenti...,Portugeese
...,...,...
495,Parken blev oprettet i 2012 for at «bevare et ...,Danish
496,Nationalparken ligger i Trysil kommune; og den...,Danish
497,Panserskibet Kaiser Max var en del af en serie...,Danish
498,Det var bygget af træ og derefter beklædt med ...,Danish


In [161]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 844 entries, 0 to 843
Data columns (total 2 columns):
 #   Column    Non-Null Count  Dtype
---  ------    --------------  -----
 0   Text      844 non-null    str  
 1   Language  844 non-null    str  
dtypes: str(2)
memory usage: 13.3 KB
